# Module 5, Topic 3 — Vector Similarity Search

**Generative AI Fellowship — Beginner**

In this notebook, we open up what "closeness" between vectors actually means — by computing it **by hand** first, then using it to search over a small set of Nigerian fintech customer-support queries, and finally watching search slow down as the dataset grows.

**What we'll do:**
1. Compute cosine similarity between two vectors, by hand
2. Compute Euclidean distance between the same two vectors, by hand
3. Build a small customer-support query corpus
4. Turn each query into a sentence vector
5. Write a brute-force nearest-neighbour search, from scratch
6. Compare cosine similarity vs. Euclidean distance rankings
7. Time the search as the number of stored documents grows

> 💡 We deliberately write the similarity math and the search loop ourselves in this notebook — no `most_similar()` shortcut this time — so the mechanics from the slides are fully visible in code.

## Step 1 — Install what we need

`numpy` gives us fast vector math. `gensim` lets us reuse the same kind of word-vector training from Topic 2 to turn our support queries into vectors.

In [ ]:
!pip install gensim numpy --quiet

## Step 2 — Import what we need

In [ ]:
import numpy as np
import time
from gensim.models import Word2Vec
from pprint import pprint

print("Ready to go!")

## Step 3 — Cosine similarity, by hand

Recall from the slides:

```
cosine_similarity(A, B) = (A · B) / (|A| × |B|)
```

Let's compute this ourselves on two small example vectors — no library function, just the formula.

In [ ]:
def cosine_similarity(a, b):
    dot_product = np.dot(a, b)
    magnitude_a = np.linalg.norm(a)
    magnitude_b = np.linalg.norm(b)
    return dot_product / (magnitude_a * magnitude_b)


vector_a = np.array([1.0, 2.0, 3.0])
vector_b = np.array([2.0, 4.0, 6.0])   # same direction as A, different length
vector_c = np.array([-1.0, -2.0, -3.0])  # opposite direction to A

print("cosine_similarity(A, B) =", cosine_similarity(vector_a, vector_b))
print("cosine_similarity(A, C) =", cosine_similarity(vector_a, vector_c))

**What to notice:** `A` and `B` point in exactly the same direction (B is just A scaled up) — cosine similarity gives `1.0`, a perfect match, even though the vectors aren't equal. `A` and `C` point in exactly opposite directions — cosine similarity gives `-1.0`. This confirms cosine similarity only cares about *angle*, not length, exactly as described on Slide 5.

## Step 4 — Euclidean distance, by hand

Now the straight-line "ruler" distance between the same vectors:

```
distance(A, B) = √[ Σ (Aᵢ - Bᵢ)² ]
```

In [ ]:
def euclidean_distance(a, b):
    return np.linalg.norm(a - b)


print("euclidean_distance(A, B) =", euclidean_distance(vector_a, vector_b))
print("euclidean_distance(A, C) =", euclidean_distance(vector_a, vector_c))

**What to notice:** Even though `A` and `B` point in the exact same direction (cosine similarity said `1.0`, a perfect match), their Euclidean distance is *not* zero — because `B` is longer than `A`. This is exactly the magnitude-sensitivity called out on Slide 4, and exactly why text embeddings usually prefer cosine similarity over Euclidean distance.

## Step 5 — A small customer-support query corpus

Below are 16 example customer-support queries for a Nigerian fintech app, grouped (informally) into 8 topics: PIN reset, failed transfers, checking balance, funding a wallet, declined cards, contacting support, BVN verification, and app crashes.

In [ ]:
support_queries = [
    "how do i reset my transaction pin",
    "i forgot my pin how do i change it",
    "my transfer failed please help",
    "why did my transfer fail",
    "how do i check my account balance",
    "i want to know my current balance",
    "how do i fund my wallet",
    "how can i add money to my wallet",
    "my card was declined at the pos",
    "why was my card declined",
    "how do i contact customer support",
    "i need to speak to an agent",
    "how do i verify my bvn",
    "what is bvn verification for",
    "my app keeps crashing",
    "the app is not opening on my phone",
]

tokenized_queries = [query.split() for query in support_queries]

print(f"{len(support_queries)} queries loaded")
print("Example:", support_queries[0], "->", tokenized_queries[0])

## Step 6 — Turn words into vectors

Just like in Topic 2, we train a small Word2Vec model — this time on our support-query corpus — so every word has a vector.

In [ ]:
word_model = Word2Vec(
    sentences=tokenized_queries,
    vector_size=30,
    window=4,
    min_count=1,
    sg=1,
    epochs=300,
    seed=42,
    workers=1,
)

print("Vocabulary size:", len(word_model.wv.key_to_index))

## Step 7 — Turn queries into sentence vectors

A word vector represents one word. To compare whole *queries*, we need one vector per sentence.

The simplest approach: average together the vectors of every word in the sentence. This isn't the most sophisticated way to build a sentence embedding, but it's enough to demonstrate search — and it's a natural bridge from Topic 2's word vectors to whole-sentence comparison.

In [ ]:
def sentence_vector(sentence, model):
    words = sentence.split()
    word_vectors = [model.wv[word] for word in words if word in model.wv]
    return np.mean(word_vectors, axis=0)


query_vectors = [sentence_vector(query, word_model) for query in support_queries]

print("Number of sentence vectors:", len(query_vectors))
print("Shape of one sentence vector:", query_vectors[0].shape)

## Step 8 — Brute-force nearest-neighbour search, from scratch

Now let's implement exactly what Slide 8 described: take a new query, compare it against **every** stored vector, and return the closest matches.

New query to search for: `"i can't remember my pin"`

In [ ]:
new_query = "i cant remember my pin"
new_query_vector = sentence_vector(new_query, word_model)

results = []
for i in range(len(support_queries)):
    score = cosine_similarity(new_query_vector, query_vectors[i])
    results.append((score, support_queries[i]))

results.sort(key=lambda pair: pair[0], reverse=True)  # highest similarity first

print(f"Query: \"{new_query}\"\n")
print("Top 5 matches (cosine similarity):")
for score, query in results[:5]:
    print(f"  {score:.3f}  -  {query}")

**What to notice:** the loop above touches every single stored vector, one at a time, and computes a similarity score for each — this is brute-force search, in full. The top two results should be the two truly PIN-related queries — a real, correct signal.

You'll also notice most of the scores sit very close together, near `1.0`. This is a real side effect of *averaging* every word's vector equally: short sentences share common connecting words ("how", "do", "i", "my"), and those words end up dominating the average, squeezing every sentence vector toward a similar overall direction. It's the same "small corpus, rough result" caveat from Topic 2 — production systems typically use purpose-built sentence embedding models (or down-weight common words) to avoid exactly this.

## Step 9 — Cosine similarity vs. Euclidean distance: do they agree?

Let's run the exact same search, but ranking by Euclidean distance instead of cosine similarity, and compare the two rankings side by side.

In [ ]:
euclidean_results = []
for i in range(len(support_queries)):
    distance = euclidean_distance(new_query_vector, query_vectors[i])
    euclidean_results.append((distance, support_queries[i]))

euclidean_results.sort(key=lambda pair: pair[0])  # smallest distance first

print("Top 5 matches (Euclidean distance):")
for distance, query in euclidean_results[:5]:
    print(f"  {distance:.3f}  -  {query}")

print("\nTop 5 matches (cosine similarity), for comparison:")
for score, query in results[:5]:
    print(f"  {score:.3f}  -  {query}")

**What to look for:** don't be surprised if the two rankings look completely different — that's the point, not a bug. Euclidean distance is sensitive to each vector's *magnitude* (its overall length), and with averaged sentence vectors, magnitude differences are mostly just noise from how many words got averaged together — not meaningful differences in topic. Cosine similarity ignores magnitude entirely and only looks at direction, which is why it lined up correctly with the true PIN-related queries above. This is a live, working demonstration of exactly why cosine similarity is the standard default for text search, as described on Slides 5 and 7.

## Step 10 — Timing brute-force search as the dataset grows

Our corpus has 16 queries — brute-force search over it is instant. Let's simulate what happens with far more stored documents, by generating larger sets of random vectors of the same shape and timing the exact same brute-force loop against each size.

In [ ]:
def time_brute_force_search(num_documents, vector_dim=30):
    rng = np.random.default_rng(seed=42)
    documents = rng.normal(size=(num_documents, vector_dim))
    query = rng.normal(size=vector_dim)

    start_time = time.time()
    scores = []
    for i in range(num_documents):
        scores.append(cosine_similarity(query, documents[i]))
    scores.sort(reverse=True)
    elapsed = time.time() - start_time

    return elapsed


dataset_sizes = [16, 1_000, 10_000, 100_000]

for size in dataset_sizes:
    elapsed_seconds = time_brute_force_search(size)
    print(f"{size:>8,} documents  ->  {elapsed_seconds:.4f} seconds")

**What to notice:** search time grows roughly in a straight line with the number of stored documents — exactly as predicted on Slide 10. At 16 documents (our actual corpus) it's instant. At 100,000 documents, the same brute-force approach is already noticeably slower — and a real Nigerian platform might store millions of documents, not thousands.

## Recap

In this notebook, we:
- Computed cosine similarity and Euclidean distance by hand, and saw why they can disagree
- Built sentence vectors for a set of customer-support queries by averaging word vectors
- Wrote a brute-force nearest-neighbour search loop from scratch and used it to find the closest matches to a new query
- Compared cosine similarity and Euclidean distance rankings on the same data
- Measured how brute-force search time grows as the number of stored documents increases

**Up next (Topic 4):** we'll stop comparing against every single document by hand, and instead store our vectors in a real **vector database** — starting with ChromaDB and FAISS — which use indexing to avoid this scaling problem entirely.